# FIFA World Cup 2026 Player Performance Analytics

## Program
### 1. Python – Data Cleaning & Transformation
### 2. MySQL – Database and Data Loading
### 3. Power BI – DAX Measures

## 1. Python – Data Cleaning & Transformation

In [14]:
import pandas as pd
from sqlalchemy import create_engine, text

# Load dataset
df = pd.read_csv("fifa_world_cup_2026_player_performance.csv")

# Create a copy for processing
fifa = df.copy()

# Handle missing values
fifa["player_name"] = fifa["player_name"].fillna("Unknown")
fifa["nationality"] = fifa["nationality"].fillna("Unknown")
fifa["team"] = fifa["team"].fillna("Unknown")
fifa["position"] = fifa["position"].fillna("Unknown")

# Remove duplicate records
fifa.drop_duplicates(inplace=True)

# Convert match date
fifa["match_date"] = pd.to_datetime(fifa["match_date"])

# Create derived columns
fifa["goal_contribution"] = (
    fifa["goals"] + fifa["assists"]
)

fifa["pass_efficiency"] = (
    fifa["successful_passes"] /
    fifa["total_passes"]
)

# Standardize column names
fifa.columns = (
    fifa.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

print("Original Shape:", df.shape)
print("Cleaned Shape:", fifa.shape)

fifa.head()

Original Shape: (54600, 75)
Cleaned Shape: (54600, 77)


,player_id,player_name,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,...,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating,goal_contribution,pass_efficiency
0,P00055,Rodri Fati,26,Spanish,Spain,3,Goalkeeper,195,75,Left,...,55.9,42.0,51.8,0,0,242,0,5.8,0,0.576923
1,P00070,Ansu Le Normand,19,Spanish,Spain,18,Midfielder,178,75,Right,...,43.7,31.1,52.7,0,3,342,0,5.5,0,0.875000
2,P00066,Gavi Ramos,18,Spanish,Spain,14,Midfielder,177,72,Left,...,99.0,83.4,54.8,1,1,245,0,8.4,1,0.847059
3,P00073,Pedro Cubarsi,20,Spanish,Spain,21,Forward,182,74,Right,...,42.3,40.9,78.5,5,3,422,0,6.7,2,0.631579
4,P00059,Alvaro Oyarzabal,23,Spanish,Spain,7,Defender,191,81,Left,...,33.5,60.0,56.6,0,0,440,0,5.7,0,0.750000


## 2. MySQL – Database and Data Loading

In [15]:
username = "root"
password = ""  # MySQL root password
host = "localhost"
port = 3306

# Connect to MySQL server and ensure database exists
server_engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}")
with server_engine.connect() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS fifa_analytics;"))
    conn.commit()

# Connect to MySQL fifa_analytics database
db_engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/fifa_analytics"
)

# Load cleaned data into MySQL
fifa.to_sql(
    "player_performance",
    con=db_engine,
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("Data loaded successfully into MySQL.")

Data loaded successfully into MySQL.


### Verify MySQL Data Loading

In [16]:
pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM player_performance",
    db_engine
)

,total_rows
0,54600


## 3. Power BI – DAX Measures

```DAX
// Total Players
Total Players =
DISTINCTCOUNT(player_performance[player_id])

// Total Goals
Total Goals =
SUM(player_performance[goals])

// Total Assists
Total Assists =
SUM(player_performance[assists])

// Average Player Rating
Average Rating =
AVERAGE(player_performance[player_rating])

// Average Performance Score
Average Performance =
AVERAGE(player_performance[performance_score])

// Total Player of Match Awards
Player of Match Awards =
SUM(player_performance[player_of_match_awards])
```